# Time of Emergence — Two Framings
## NH September Sea Ice Area  |  G6-1.5K-HiLLA vs SSP2-4.5

Two complementary detection methods on the same physical question:

- **Perfect-model (left):** MIROC-ES2H 10-member ensembles fully known — Welch's t-test on per-member window means detects when the forced responses first diverge.
- **Obs-only (right):** MIROC-ES2H single realisation — CMIP6 historical r1 (1850–2014) joined to SSP2-4.5 r1 (2015–2034) trains the AR(1) model; plot shows from 1980; OLS linear trend + AR(1) residuals extrapolates the counterfactual; ToE when HiLLA r1 exits the 95% prediction interval for 5 consecutive years.

Grid-cell areas from MIROC6 `areacella` file (m²), loaded from disk.


In [7]:
import fsspec
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import matplotlib
matplotlib.rcParams.update({'font.size': 13})


In [8]:
# ── MIROC-ES2H (both panels) ─────────────────────────────────────────────────
_MIROC = 's3://reflective-persistent-prod-large/MIROC-ES2H'

miroc_HiLLA_files  = [
    f'{_MIROC}/G6-1.5K-HiLLA/Omon/siconc_G6-1.5K-HiLLA_r{i:02d}.nc'
    for i in range(1, 11)
]
miroc_ssp245_files = [
    f'{_MIROC}/G6-1.5K-HiLLA/Omon/siconc_baseline_r{i:02d}.nc'
    for i in range(1, 11)
]

# Historical r1 — used as the 'observed' counterfactual record (right panel)
miroc_hist_r1_file = (
    f'{_MIROC}/historical/SImon/'
    'siconc_SImon_MIROC-ES2H_historical_r1i1p4f2_gn_185001-201412.nc'
)

# Grid-cell areas from MIROC6 (same T85 atm grid as MIROC-ES2H), on disk
AREACELLA_FILE = (
    '/home/jovyan/shared/Data/model_grids_fx/'
    'areacella_fx_MIROC6_historical_r1i1p1f1_gn.nc'
)


In [9]:
def read_miroc_ice(file_list):
    """Load MIROC-ES2H siconc: rename non-standard lat dim; fraction → %."""
    # Auxiliary dims that are never the spatial lat/lon grid dimensions
    _AUX_DIMS = {'bnds', 'nbnd', 'nbnds', 'vertices', 'nverts', 'd2'}

    ds_list = []
    for f in file_list:
        with fsspec.open(f, mode='rb') as fobj:
            ds = xr.open_dataset(fobj, engine='h5netcdf', chunks={}).load()

        # Rename the non-standard latitude dimension → 'lat'
        # Skip auxiliary dims (bnds, vertices) which appear in the dim list
        # but are not the spatial latitude axis
        extra = [d for d in ds.dims
                 if d not in ('lon', 'time', 'lat') and d not in _AUX_DIMS]
        if extra:
            ds = ds.rename({extra[0]: 'lat'})

        # Identify the sea-ice concentration variable by name
        # Exclude vertex variables (e.g. vertices_latitude) — 'ice' is a
        # substring of 'vertices' (v-e-r-t-ice-s), which caused false matches
        ice_var = next(
            (v for v in ds.data_vars
             if ('ice' in v.lower() or 'conc' in v.lower() or v == 'SeaIce')
             and 'vertices' not in v.lower()),
            None
        )
        if ice_var and ice_var != 'siconc':
            ds = ds.rename({ice_var: 'siconc'})

        if 'siconc' in ds and float(ds['siconc'].max()) <= 1.5:
            ds['siconc'] = ds['siconc'] * 100.0
        ds_list.append(ds)
    return ds_list


# Load MIROC 10-member ensembles (left panel)
miroc_HiLLA  = read_miroc_ice(miroc_HiLLA_files)
miroc_ssp245 = read_miroc_ice(miroc_ssp245_files)
print(f'MIROC HiLLA:  {len(miroc_HiLLA)} members  | '
      f'{miroc_HiLLA[0].time.values[[0,-1]]}')
print(f'MIROC SSP245: {len(miroc_ssp245)} members  | '
      f'{miroc_ssp245[0].time.values[[0,-1]]}')

# Load MIROC historical r1 (right panel: 'observed' counterfactual)
miroc_hist_r1_ds = read_miroc_ice([miroc_hist_r1_file])[0]
print(f'MIROC hist r1: {miroc_hist_r1_ds.time.values[[0,-1]]}')

# Load MIROC6 areacella (grid-cell areas in m²)
areacella_ds = xr.open_dataset(AREACELLA_FILE)
areacella    = areacella_ds['areacella']  # (lat, lon), m²
print(f'areacella: dims={dict(areacella.sizes)}')


MIROC HiLLA:  10 members  | ['2035-01-16T12:00:00.000000000' '2084-12-16T12:00:00.000000000']
MIROC SSP245: 10 members  | ['2020-01-16T12:00:00.000000000' '2084-12-16T12:00:00.000000000']
MIROC hist r1: ['1850-01-16T12:00:00.000000000' '2014-12-16T12:00:00.000000000']
areacella: dims={'lat': 128, 'lon': 256}


In [10]:
R_EARTH = 6.371e6  # m


def _area_from_vertices(ds):
    """Spherical cell area (m²) from CMIP6 vertices_latitude/longitude.

    Spherical shoelace: A = R²/2 |Σᵢ Δλᵢ (sin φᵢ₊₁ + sin φᵢ)|
    Handles longitude wrapping; works for regular and curvilinear cells.
    """
    vlat = np.deg2rad(ds['vertices_latitude'].values)   # (y, x, 4)
    vlon = np.deg2rad(ds['vertices_longitude'].values)  # (y, x, 4)
    area = np.zeros(vlat.shape[:2])
    for i in range(4):
        j = (i + 1) % 4
        dlon = vlon[..., j] - vlon[..., i]
        dlon = (dlon + np.pi) % (2 * np.pi) - np.pi   # wrap to [-π, π]
        area += dlon * (np.sin(vlat[..., j]) + np.sin(vlat[..., i]))
    return R_EARTH**2 * np.abs(area) / 2  # (y, x) m²


def september_sia(ds_list, area_da=None, lat_min=0.0):
    """NH September SIA (10⁶ km²) — handles regular and curvilinear grids.

    Curvilinear (vertices_latitude present): area from vertices;
      NH selection via 2D latitude coordinate.
    Regular lat/lon: area from area_da (areacella DataArray, required).
    """
    if area_da is not None:
        a_lat = [d for d in area_da.dims if 'lat' in d.lower()][0]
        if float(area_da[a_lat][0]) > float(area_da[a_lat][-1]):
            area_da = area_da.isel(**{a_lat: slice(None, None, -1)})
        area_nh_reg = area_da.sel(**{a_lat: slice(lat_min, 90)}).values
    else:
        area_nh_reg = None

    result = []
    for ds in ds_list:
        da = ds['siconc']

        if 'vertices_latitude' in ds:
            # ── Curvilinear ocean grid ──────────────────────────────────────
            area_2d    = _area_from_vertices(ds)         # (y, x) m²
            lat_2d     = ds['latitude'].values           # (y, x)
            area_nh_2d = np.where(lat_2d >= lat_min, area_2d, 0.0)

            sc = np.where(np.isnan(da.values), 0.0, da.values)
            sia_arr   = (sc / 100.0 * area_nh_2d).sum(axis=(-2, -1)) / 1e12
            time_vals = da.time.values
        else:
            # ── Regular lat/lon grid ────────────────────────────────────────
            lat_nm = 'lat' if 'lat' in da.dims else 'latitude'
            if float(da[lat_nm][0]) > float(da[lat_nm][-1]):
                da = da.isel(**{lat_nm: slice(None, None, -1)})
            da_nh = da.sel(**{lat_nm: slice(lat_min, 90)})
            sc = np.where(np.isnan(da_nh.values), 0.0, da_nh.values)
            sia_arr   = (sc / 100.0 * area_nh_reg).sum(axis=(-2, -1)) / 1e12
            time_vals = da_nh.time.values

        sia_da = xr.DataArray(sia_arr, coords={'time': time_vals}, dims=['time'])
        sep = sia_da.sel(time=sia_da.time.dt.month == 9)
        result.append(sep.assign_coords(time=sep.time.dt.year.values.astype(int)))
    return result


# MIROC 10-member ensembles (regular grid → uses areacella)
sep_miroc_HiLLA  = september_sia(miroc_HiLLA,  areacella)
sep_miroc_ssp245 = september_sia(miroc_ssp245, areacella)

# MIROC historical r1 (curvilinear ocean grid → uses vertex areas internally)
sep_miroc_hist_r1   = september_sia([miroc_hist_r1_ds], areacella)[0]
sep_miroc_ssp245_r1 = sep_miroc_ssp245[0]   # member 1 of already-loaded list
sep_miroc_HiLLA_r1  = sep_miroc_HiLLA[0]    # member 1 of already-loaded list

# Training record: MIROC historical (1850–2014) + SSP245 r1 (2015–2034)
SAT_START  = 1980
START_YEAR = 2035
miroc_obs = xr.concat([
    sep_miroc_hist_r1.sel(time=slice(1850, 2014)),
    sep_miroc_ssp245_r1.sel(time=slice(2015, 2034)),
], dim='time')

print(f'MIROC HiLLA r1:  years {sep_miroc_HiLLA_r1.time.values[[0,-1]]}  '
      f'mean = {float(sep_miroc_HiLLA_r1.mean()):.2f} Mkm²')
print(f'MIROC obs record: {miroc_obs.time.values[[0,-1]]}  '
      f'({len(miroc_obs)} years)')


ValueError: operands could not be broadcast together with shapes (600,129,360) (64,256) 

In [ ]:
def read_miroc_ice(file_list):
    """Load MIROC-ES2H siconc: rename non-standard lat dim; fraction → %."""
    ds_list = []
    for f in file_list:
        with fsspec.open(f, mode='rb') as fobj:
            ds = xr.open_dataset(fobj, engine='h5netcdf', chunks={}).load()
        extra = [d for d in ds.dims if d not in ('lon', 'time') and d != 'lat']
        if extra:
            ds = ds.rename({extra[0]: 'lat'})
        ice_var = next(
            (v for v in ds.data_vars
             if 'ice' in v.lower() or 'conc' in v.lower() or v == 'SeaIce'), None
        )
        if ice_var and ice_var != 'siconc':
            ds = ds.rename({ice_var: 'siconc'})
        if 'siconc' in ds and float(ds['siconc'].max()) <= 1.5:
            ds['siconc'] = ds['siconc'] * 100.0
        ds_list.append(ds)
    return ds_list


# Load MIROC 10-member ensembles (left panel)
miroc_HiLLA  = read_miroc_ice(miroc_HiLLA_files)
miroc_ssp245 = read_miroc_ice(miroc_ssp245_files)
print(f'MIROC HiLLA:  {len(miroc_HiLLA)} members  | '
      f'{miroc_HiLLA[0].time.values[[0,-1]]}')
print(f'MIROC SSP245: {len(miroc_ssp245)} members  | '
      f'{miroc_ssp245[0].time.values[[0,-1]]}')

# Load MIROC historical r1 (right panel: 'observed' counterfactual)
miroc_hist_r1_ds = read_miroc_ice([miroc_hist_r1_file])[0]
print(f'MIROC hist r1: {miroc_hist_r1_ds.time.values[[0,-1]]}')

# Load MIROC6 areacella (grid-cell areas in m²)
areacella_ds = xr.open_dataset(AREACELLA_FILE)
areacella    = areacella_ds['areacella']  # (lat, lon), m²
print(f'areacella: dims={dict(areacella.sizes)}')


In [ ]:
def september_sia(ds_list, area_da, lat_min=0.0):
    """NH September SIA (10⁶ km²) using provided grid-cell areas (m²)."""
    # Sort area_da to ascending lat; extract NH values once
    a_lat = [d for d in area_da.dims if 'lat' in d.lower()][0]
    a_lon = [d for d in area_da.dims if 'lon' in d.lower()][0]
    if float(area_da[a_lat][0]) > float(area_da[a_lat][-1]):
        area_da = area_da.isel(**{a_lat: slice(None, None, -1)})
    area_nh = area_da.sel(**{a_lat: slice(lat_min, 90)}).values  # (n_lat_nh, n_lon)

    result = []
    for ds in ds_list:
        da = ds['siconc']
        lat_nm = 'lat' if 'lat' in da.dims else 'latitude'
        lon_nm = 'lon' if 'lon' in da.dims else 'longitude'
        if float(da[lat_nm][0]) > float(da[lat_nm][-1]):
            da = da.isel(**{lat_nm: slice(None, None, -1)})
        da_nh = da.sel(**{lat_nm: slice(lat_min, 90)})
        # (time, lat_nh, lon) * (lat_nh, lon) → broadcast, sum → (time,)
        sia_arr = (da_nh.values / 100.0 * area_nh).sum(axis=(-2, -1)) / 1e12
        sia_da  = xr.DataArray(sia_arr,
                               coords={'time': da_nh.time.values}, dims=['time'])
        sep = sia_da.sel(time=sia_da.time.dt.month == 9)
        result.append(sep.assign_coords(time=sep.time.dt.year.values.astype(int)))
    return result


# MIROC 10-member ensembles
sep_miroc_HiLLA  = september_sia(miroc_HiLLA,  areacella)
sep_miroc_ssp245 = september_sia(miroc_ssp245, areacella)

# MIROC r1 individual timeseries for the obs-only panel
sep_miroc_hist_r1   = september_sia([miroc_hist_r1_ds], areacella)[0]
sep_miroc_ssp245_r1 = sep_miroc_ssp245[0]   # member 1 of already-loaded list
sep_miroc_HiLLA_r1  = sep_miroc_HiLLA[0]    # member 1 of already-loaded list

# Training record: MIROC historical (1850–2014) + SSP245 r1 (2015–2034)
SAT_START  = 1980   # first year shown in obs-only panel
START_YEAR = 2035
miroc_obs = xr.concat([
    sep_miroc_hist_r1.sel(time=slice(1850, 2014)),
    sep_miroc_ssp245_r1.sel(time=slice(2015, 2034)),
], dim='time')

print(f'MIROC HiLLA r1:  years {sep_miroc_HiLLA_r1.time.values[[0,-1]]}  '
      f'mean = {float(sep_miroc_HiLLA_r1.mean()):.2f} Mkm²')
print(f'MIROC obs record: {miroc_obs.time.values[[0,-1]]}  '
      f'({len(miroc_obs)} years)')


In [ ]:
Y_MIN  = 2
Y_MAX  = 40
y_vals = np.arange(Y_MIN, Y_MAX + 1)


def perfect_model_toe(hilla_ts_list, ssp245_ts_list,
                      start_year=START_YEAR, y_vals=y_vals):
    """Welch's one-tailed t-test: per-member window-mean HiLLA > SSP245.

    For each window length Y, compute each member's mean SIA over
    [start_year, start_year+Y-1] and test HiLLA (n=10) > SSP245 (n=10).

    Returns (p_vals, toe_year):
      p_vals   — one-tailed p-value for each Y
      toe_year — last year of first significant window, or None
    """
    p_arr = np.full(len(y_vals), np.nan)
    for i, Y in enumerate(y_vals):
        h = [float(ts.sel(time=slice(start_year, start_year + int(Y) - 1)).mean())
             for ts in hilla_ts_list]
        s = [float(ts.sel(time=slice(start_year, start_year + int(Y) - 1)).mean())
             for ts in ssp245_ts_list]
        _, p = stats.ttest_ind(h, s, equal_var=False, alternative='greater')
        p_arr[i] = p
    first = np.where(p_arr < 0.05)[0]
    toe_year = int(start_year + y_vals[first[0]] - 1) if len(first) else None
    return p_arr, toe_year


pm_pvals, pm_toe = perfect_model_toe(sep_miroc_HiLLA, sep_miroc_ssp245)
wlen = (pm_toe - START_YEAR + 1) if pm_toe else None
print(f'Perfect-model ToE: {pm_toe}  (window = {wlen} yr)')

In [ ]:
PRED_YEARS = np.arange(START_YEAR, 2085)
K_CONSEC   = 5     # consecutive years above upper PI required for ToE
CI_LEVEL   = 0.95
TREND_START = 1980  # OLS fit window: declining period only


def ar1_forecast(obs_ts, pred_years,
                 trend_start=TREND_START, ci=CI_LEVEL):
    """OLS linear trend + AR(1) residuals h-step forecast.

    Steps:
      1. Fit OLS line to obs_ts from trend_start onwards.
      2. Compute residuals; fit AR(1):  ε_t = φ·ε_{t-1} + η,  η ~ N(0, σ²).
      3. h-step mean:  trend(t) + φ^h · ε_last  (AR contribution decays to 0).
      4. h-step PI:    mean ± z·σ·√((1 − φ^{2h}) / (1 − φ²)).

    Returns (mu, lower_ci, upper_ci).
    """
    mask  = obs_ts.time.values >= trend_start
    x_fit = obs_ts.time.values[mask].astype(float)
    y_fit = obs_ts.values[mask].astype(float)

    slope, intercept, *_ = stats.linregress(x_fit, y_fit)

    resid    = y_fit - (slope * x_fit + intercept)
    phi      = np.dot(resid[1:], resid[:-1]) / np.dot(resid[:-1], resid[:-1])
    innov    = resid[1:] - phi * resid[:-1]
    sigma    = np.sqrt(np.sum(innov**2) / (len(innov) - 1))
    eps_last = resid[-1]

    h   = pred_years.astype(float) - x_fit[-1]   # 1, 2, 3, ...
    mu  = slope * pred_years + intercept + phi**h * eps_last

    z       = stats.norm.ppf((1 + ci) / 2)
    pi_half = z * sigma * np.sqrt((1 - phi**(2*h)) / max(1 - phi**2, 1e-8))

    print(f'OLS slope:  {slope:.4f} Mkm²/yr  ({int(trend_start)}–{int(x_fit[-1])})')
    print(f'AR(1):  φ = {phi:.3f},  σ_η = {sigma:.3f} Mkm²')
    return mu, mu - pi_half, mu + pi_half


def ar1_toe(hilla_ts, pred_years, upper, start_year=START_YEAR, K=K_CONSEC):
    """First year where HiLLA > upper PI for K consecutive years."""
    h_yrs  = hilla_ts.time.values
    common = np.intersect1d(h_yrs[h_yrs >= start_year], pred_years)
    if len(common) < K:
        return None
    h_vals = hilla_ts.sel(time=common).values
    ui     = upper[np.searchsorted(pred_years, common)]
    above  = h_vals > ui
    for i in range(len(above) - K + 1):
        if above[i : i + K].all():
            return int(common[i])
    return None


ar1_mu, ar1_lower, ar1_upper = ar1_forecast(miroc_obs, PRED_YEARS)
obs_toe = ar1_toe(sep_miroc_HiLLA_r1, PRED_YEARS, ar1_upper)
print(f'Obs-only ToE (K={K_CONSEC}): {obs_toe}')

In [ ]:
BLUE       = '#2171B5'
BLUE_LIGHT = '#BDD7E7'
GRAY       = '#636363'
GRAY_LIGHT = '#D9D9D9'


def ts_stack(ts_list):
    """Stack year-indexed 1-D DataArrays into (n_members, n_years); common years only."""
    common = sorted(set.intersection(*[set(ts.time.values.tolist()) for ts in ts_list]))
    return np.stack([ts.sel(time=common).values for ts in ts_list]), np.array(common, dtype=int)


def plot_ens(ax, arr, years, color, light, label):
    mean = arr.mean(axis=0)
    p05  = np.percentile(arr,  5, axis=0)
    p95  = np.percentile(arr, 95, axis=0)
    ax.fill_between(years, p05, p95, color=light, alpha=0.55, zorder=1)
    ax.plot(years, mean, color=color, lw=2.2, label=label, zorder=2)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
fig.subplots_adjust(wspace=0.04)

# ── Left panel: perfect-model (MIROC) ──────────────────────────────────────────
hilla_arr, hilla_years = ts_stack(sep_miroc_HiLLA)
ssp_arr,   ssp_years   = ts_stack(sep_miroc_ssp245)

plot_ens(ax1, ssp_arr[:, ssp_years >= 2015], ssp_years[ssp_years >= 2015],
         GRAY, GRAY_LIGHT, 'SSP2-4.5  (n=10)')
plot_ens(ax1, hilla_arr, hilla_years, BLUE, BLUE_LIGHT, 'G6-1.5K-HiLLA  (n=10)')

ax1.axvline(START_YEAR, color='k', lw=1.2, ls='--', alpha=0.45, label='SAI start (2035)')

if pm_toe:
    ax1.axvline(pm_toe, color=BLUE, lw=2, ls=':', zorder=5)
    ax1.text(pm_toe + 0.5, 0.96, f'ToE = {pm_toe}',
             transform=ax1.get_xaxis_transform(),
             color=BLUE, fontsize=11, va='top', fontweight='bold')

ax1.set_xlim(2015, 2080)
ax1.set_xlabel('Year')
ax1.set_ylabel('NH September SIA  (10\u2076 km\u00b2)')
ax1.set_title('Perfect-model framing  (MIROC-ES2H)\n'
              'Known 10-member ensembles — two-sample test', fontsize=11, pad=8)
ax1.legend(fontsize=10, loc='upper right', framealpha=0.9)
ax1.grid(ls='--', alpha=0.3)
for sp in ('top', 'right'):
    ax1.spines[sp].set_visible(False)

# ── Right panel: obs-only (MIROC-ES2H) ───────────────────────────────────────
# Observed record: plot from SAT_START (AR(1) trained on 1980–2034 trend segment)
_obs_plot = miroc_obs.sel(time=slice(SAT_START, 2034))
ax2.plot(_obs_plot.time.values, _obs_plot.values,
         color=GRAY, lw=2.2, label='"Observed"  (historical + SSP2-4.5 r1)')

# AR(1) counterfactual: mean + 95 % PI
ax2.fill_between(PRED_YEARS, ar1_lower, ar1_upper,
                 color=GRAY_LIGHT, alpha=0.7, zorder=1,
                 label=f'AR(1) counterfactual  ({int(CI_LEVEL*100)}% PI)')
ax2.plot(PRED_YEARS, ar1_mu, color=GRAY, lw=1.8, ls='--', zorder=2)

# MIROC HiLLA r1
h_yrs = sep_miroc_HiLLA_r1.time.values[sep_miroc_HiLLA_r1.time.values <= 2080]
ax2.plot(h_yrs, sep_miroc_HiLLA_r1.sel(time=h_yrs).values,
         color=BLUE, lw=2.2, label='G6-1.5K-HiLLA  r1', zorder=3)

ax2.axvline(START_YEAR, color='k', lw=1.2, ls='--', alpha=0.45, label='SAI start  (2035)')

if obs_toe:
    ax2.axvline(obs_toe, color=BLUE, lw=2, ls=':', zorder=5)
    ax2.text(obs_toe + 0.5, 0.96, f'ToE = {obs_toe}\n(K={K_CONSEC} yr)',
             transform=ax2.get_xaxis_transform(),
             color=BLUE, fontsize=11, va='top', fontweight='bold')

ax2.set_xlim(1975, 2080)
ax2.set_xlabel('Year')
ax2.set_ylabel('NH September SIA  (10\u2076 km\u00b2)')
ax2.set_title('Obs-only framing  (MIROC-ES2H)\n'
              'Single realisation — linear trend + AR(1) counterfactual', fontsize=11, pad=8)
ax2.legend(fontsize=10, loc='lower left', framealpha=0.9)
ax2.grid(ls='--', alpha=0.3)
for ax in ax1, ax2:
    ax.set_ylim(0, None)
for sp in ('top', 'right'):
    ax2.spines[sp].set_visible(False)

fig.suptitle(
    'Time of Emergence  |  NH September Sea Ice Area  |  G6-1.5K-HiLLA vs SSP2-4.5',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig('Figures/ToE_two_panel.png', dpi=200, bbox_inches='tight')
plt.show()